# Día 4 — SQL avanzado: funciones de ventana y el gap ladder en SQL
### Preparación entrevista técnica Ebury (Treasury/ALM)

Hoy replicamos en SQL lo mismo que construiste en pandas el Día 3: buckets de plazo y un gap ladder acumulado. Cada ejercicio lo harás **dos veces** -- con `sqlite3` (por si te dan un archivo de base de datos) y con DuckDB (por si trabajas sobre un DataFrame) -- para que no dependas de adivinar cuál te van a dar el lunes.

Usamos `tesoreria.db`, la misma base de datos del Día 2b. Asegúrate de tenerla en la misma carpeta.


In [11]:
import pandas as pd
import numpy as np
import sqlite3
import duckdb
pd.set_option('display.width', 1000)

In [12]:
conn = sqlite3.connect("tesoreria.db")

# Check the table characteristics
tablas=pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';",conn)
print(tablas) # We check number of tables

# Get shape of the tables
for table_name in tablas["name"]:
    table_sql=pd.read_sql(f"SELECT * FROM {table_name};", conn)
    print(f"Table: ",{table_name}, "shape:" ,{table_sql.shape})
    print(table_sql.dtypes)
    print(f"Table: {table_name}")
    print(pd.read_sql(f"SELECT * FROM {table_name};",conn).head())



             name
0    transactions
1  counterparties
Table:  {'transactions'} shape: {(41, 6)}
TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes              object
dtype: object
Table: transactions
   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
Table:  {'counterparties'} shape: {(5, 3)}
CounterpartyID    int64
Name                str
ClientType          str
dtype: object
Table: counterparties
   CounterpartyID         Name ClientType
0               1    Acme Corp  Corporate
1               2    Beta PYME     

## Parte 1 — Repaso relámpago de sintaxis básica

Por si necesitas refrescarlo en 30 segundos antes de la prueba:

```sql
SELECT columna1, SUM(columna2) as total
FROM tabla
WHERE condicion
GROUP BY columna1
HAVING SUM(columna2) > 1000    -- filtra DESPUÉS de agrupar (WHERE filtra antes)
ORDER BY total DESC
LIMIT 10
```

**Orden real de ejecución en tu cabeza (no el orden en que se escribe)**: `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`. Por eso `HAVING` puede filtrar sobre `SUM(columna2)` (ya calculado) y `WHERE` no.

### Ejercicio 1.1
Escribe una consulta que devuelva las divisas cuyo importe total (en valor absoluto) supere 300.000, usando `HAVING`.

In [88]:
# TODO
query11="""
SELECT Currency, SUM(Amount) as sum_tot
FROM transactions
WHERE Amount IS NOT NULL
GROUP BY Currency
HAVING ABS(sum_tot)>300000
ORDER BY sum_tot DESC
"""
result11=pd.read_sql(query11,conn)
print(result11)


  Currency  tot_amount
0      JPY   350667.26
1      usd  -605225.23
  Currency    sum_tot
0      JPY  350667.26
1      usd -605225.23


**Solución 1.1**

In [8]:
query = """
SELECT Currency, SUM(Amount) as total_amount
FROM transactions
WHERE Amount IS NOT NULL
GROUP BY Currency
HAVING ABS(SUM(Amount)) > 300000
"""
resultado = pd.read_sql(query, conn)
print(resultado)


  Currency  total_amount
0      JPY     350667.26
1      usd    -605225.23


## Parte 2 — `CASE WHEN`: el equivalente SQL de `pd.cut()`

Antes de poder construir un gap ladder en SQL, necesitas clasificar cada transacción en un bucket de plazo -- igual que hiciste con `pd.cut()` en pandas. En SQL, esto se hace con `CASE WHEN`.

In [13]:
query = """
SELECT
    TransID,
    Currency,
    Amount,
    MaturityDate,
    CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) as dias_hasta_vencimiento,
    CASE
        WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 7 THEN '0-7d'
        WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 30 THEN '7-30d'
        WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 90 THEN '30-90d'
        WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 180 THEN '90-180d'
        ELSE '>180d'
    END as tenor_bucket
FROM transactions
WHERE Amount IS NOT NULL
LIMIT 10
"""
resultado = pd.read_sql(query, conn)
print(resultado)


   TransID Currency     Amount         MaturityDate  dias_hasta_vencimiento tenor_bucket
0     1001      JPY   36965.83  2026-09-02 00:00:00                       7         0-7d
1     1002      usd -181419.83  2026-11-29 00:00:00                      95      90-180d
2     1003      GBP   43017.94  2026-06-12 00:00:00                     -75         0-7d
3     1004      usd -131790.35  2026-09-23 00:00:00                      28        7-30d
4     1005      usd -173979.36  2026-11-16 00:00:00                      82       30-90d
5     1007      GBP  186252.81  2026-09-11 00:00:00                      16        7-30d
6     1008      GBP  123358.94  2027-12-01 00:00:00                     462        >180d
7     1009      GBP  -78154.49  2026-12-29 00:00:00                     125      90-180d
8     1010      usd -160931.15  2026-12-31 00:00:00                     127      90-180d
9     1011      JPY   73693.21  2026-11-25 00:00:00                      91      90-180d


> `julianday()` es específico de SQLite, para restar fechas (convierte una fecha a un número y permite restarlas). En PostgreSQL harías simplemente `MaturityDate - DATE '2026-08-26'`; en la mayoría de motores modernos restar dos fechas ya da directamente el número de días. Este es exactamente el tipo de diferencia "de motor" que comentamos que no vale la pena memorizar al detalle -- lo importante es la lógica del `CASE WHEN`, no la función de fecha concreta.

### Ejercicio 2.1
Usando `CASE WHEN`, crea una columna `tipo` con el valor `"entrada"` si `Amount > 0` y `"salida"` en caso contrario -- el equivalente SQL de `np.where()` que usaste en pandas.

In [90]:
# TODO
query_tipo21 = """
SELECT Amount,
    CASE
        WHEN Amount > 0 THEN 'entrada'
        ELSE 'salida'
    END as tipo
FROM transactions
WHERE Amount IS NOT NULL
LIMIT 10
"""
resultado21 = pd.read_sql(query_tipo21, conn)
print(resultado21)


      Amount     tipo
0   36965.83  entrada
1 -181419.83   salida
2   43017.94  entrada
3 -131790.35   salida
4 -173979.36   salida
5  186252.81  entrada
6  123358.94  entrada
7  -78154.49   salida
8 -160931.15   salida
9   73693.21  entrada
      Amount     tipo
0   36965.83  entrada
1 -181419.83   salida
2   43017.94  entrada
3 -131790.35   salida
4 -173979.36   salida
5  186252.81  entrada
6  123358.94  entrada
7  -78154.49   salida
8 -160931.15   salida
9   73693.21  entrada


**Solución 2.1**

In [ ]:
query_tipo = """
SELECT TransID, Currency, Amount,
    CASE WHEN Amount > 0 THEN 'entrada' ELSE 'salida' END as tipo
FROM transactions
WHERE Amount IS NOT NULL
LIMIT 10
"""
resultado = pd.read_sql(query_tipo, conn)
print(resultado)


## Parte 3 — Funciones de ventana: el equivalente SQL de `cumsum()`

Esta es la pieza que estabas buscando el otro día. `SUM(...) OVER (PARTITION BY ... ORDER BY ...)` calcula una suma acumulada, exactamente como `cumsum()` en pandas.

In [20]:
query = """
SELECT
    Currency,
    MaturityDate,
    Amount,
    SUM(Amount) OVER (PARTITION BY Currency ORDER BY MaturityDate) as acumulado
FROM transactions
WHERE Amount IS NOT NULL
ORDER BY Currency, MaturityDate
LIMIT 15
"""
resultado = pd.read_sql(query, conn)
print(resultado)


   Currency         MaturityDate     Amount  acumulado
0       NaN  2026-09-20 00:00:00 -151184.71 -151184.71
1       EUR  2026-09-24 00:00:00  -69867.87  -69867.87
2       EUR  2026-10-20 00:00:00 -186244.59 -256112.46
3       EUR  2026-11-18 00:00:00  -75315.57 -331428.03
4       EUR  2027-08-03 00:00:00  110053.13 -221374.90
5       GBP  2026-06-12 00:00:00   43017.94   43017.94
6       GBP  2026-09-11 00:00:00  186252.81  229270.75
7       GBP  2026-10-21 00:00:00 -121606.86  107663.89
8       GBP  2026-11-14 00:00:00   17078.43  124742.32
9       GBP  2026-12-29 00:00:00  -78154.49   46587.83
10      GBP  2026-12-31 00:00:00 -181909.08 -135321.25
11      GBP  2027-02-18 00:00:00  -23939.00 -159260.25
12      GBP  2027-12-01 00:00:00  123358.94  -35901.31
13      JPY  2026-09-02 00:00:00   36965.83   36965.83
14      JPY  2026-09-13 00:00:00 -143630.31 -106664.48


**Cómo leer esto**: `PARTITION BY Currency` = "reinicia el acumulado para cada divisa" (equivalente a `groupby("divisa")` antes de `cumsum()`). `ORDER BY MaturityDate` dentro del `OVER` = "acumula en este orden" (equivalente a ordenar por fecha antes de aplicar `cumsum()` en pandas). Es exactamente el mismo concepto, dos sintaxis distintas.

Otras funciones de ventana útiles que puedes encontrarte:
- `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` -- numera las filas dentro de cada grupo (1, 2, 3...).
- `RANK() OVER (...)` -- como ROW_NUMBER, pero da el mismo número a empates.
- `AVG(...) OVER (...)` -- media dentro de la ventana, en vez de suma.

### Ejercicio 3.1 (el gap ladder completo, en SQL)
Junta todo: clasifica cada transacción en un `tenor_bucket` con `CASE WHEN`, agrupa por divisa y bucket, y calcula el acumulado con una función de ventana. Pista: en SQL no puedes anidar directamente un `GROUP BY` y luego un `OVER` en la misma consulta sin subconsulta -- necesitas dos pasos: (1) una subconsulta que agregue por bucket y divisa, (2) la consulta externa que calcule el acumulado sobre ese resultado ya agregado.

In [41]:
# TODO -- pista: usa una subconsulta (WITH ... AS, o un SELECT anidado en el FROM)


query_gap_ladder = """
SELECT Currency,
    CASE
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=7 THEN '0-7d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=30 THEN '7-30d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=90 THEN '30-90d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=180 THEN '90-180d'
        ELSE '>180d'
    END as tenor_bucket,
    SUM(Amount) OVER (
    PARTITION BY Currency,
       CASE
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=7 THEN '0-7d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=30 THEN '7-30d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=90 THEN '30-90d'
        WHEN CAST(julianday(MaturityDate)-julianday('2026-08-26') as INTEGER) <=180 THEN '90-180d'
        ELSE '>180d'
       END
    ORDER BY MaturityDate) as acumulado
FROM transactions
WHERE Amount IS NOT NULL
GROUP BY Currency,tenor_bucket

LIMIT 20
"""
resultado31=pd.read_sql(query_gap_ladder,conn)
print(resultado31)

# Method 2

query_gap_ladder = """
WITH clasificado AS (
    SELECT
        Currency,
        Amount,
        CASE
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 7 THEN '1-0-7d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 30 THEN '2-7-30d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 90 THEN '3-30-90d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 180 THEN '4-90-180d'
            ELSE '5->180d'
        END as tenor_bucket
    FROM transactions
    WHERE Amount IS NOT NULL
),
agregado AS (
    SELECT Currency, tenor_bucket, SUM(Amount) as neto_bucket
    FROM clasificado
    GROUP BY Currency, tenor_bucket
)
SELECT
    Currency,
    tenor_bucket,
    neto_bucket,
    SUM(neto_bucket) OVER (PARTITION BY Currency ORDER BY tenor_bucket) as gap_acumulado
FROM agregado
ORDER BY Currency, tenor_bucket
"""
resultado = pd.read_sql(query_gap_ladder, conn)
print(resultado)


   Currency tenor_bucket  acumulado
0       NaN        7-30d -151184.71
1       EUR       30-90d -186244.59
2       EUR       30-90d -261560.16
3       EUR        7-30d  -69867.87
4       EUR        >180d  110053.13
5       GBP         0-7d   43017.94
6       GBP       30-90d -121606.86
7       GBP       30-90d -104528.43
8       GBP        7-30d  186252.81
9       GBP      90-180d  -78154.49
10      GBP      90-180d -260063.57
11      GBP      90-180d -284002.57
12      GBP        >180d  123358.94
13      JPY         0-7d   36965.83
14      JPY       30-90d  187833.85
15      JPY       30-90d  351562.01
16      JPY        7-30d -143630.31
17      JPY        7-30d  -22751.52
18      JPY      90-180d   73693.21
19      JPY      90-180d  -17767.18
   Currency tenor_bucket  neto_bucket  gap_acumulado
0       NaN      2-7-30d   -151184.71     -151184.71
1       EUR      2-7-30d    -69867.87      -69867.87
2       EUR     3-30-90d   -261560.16     -331428.03
3       EUR      5->180d    1100

**Solución 3.1**

In [29]:
query_gap_ladder = """
WITH clasificado AS (
    SELECT
        Currency,
        Amount,
        CASE
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 7 THEN '1-0-7d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 30 THEN '2-7-30d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 90 THEN '3-30-90d'
            WHEN CAST(julianday(MaturityDate) - julianday('2026-08-26') AS INTEGER) <= 180 THEN '4-90-180d'
            ELSE '5->180d'
        END as tenor_bucket
    FROM transactions
    WHERE Amount IS NOT NULL
),
agregado AS (
    SELECT Currency, tenor_bucket, SUM(Amount) as neto_bucket
    FROM clasificado
    GROUP BY Currency, tenor_bucket
)
SELECT
    Currency,
    tenor_bucket,
    neto_bucket,
    SUM(neto_bucket) OVER (PARTITION BY Currency ORDER BY tenor_bucket) as gap_acumulado
FROM agregado
ORDER BY Currency, tenor_bucket
"""
resultado = pd.read_sql(query_gap_ladder, conn)
print(resultado)


   Currency tenor_bucket  neto_bucket  gap_acumulado
0       NaN      2-7-30d   -151184.71     -151184.71
1       EUR      2-7-30d    -69867.87      -69867.87
2       EUR     3-30-90d   -261560.16     -331428.03
3       EUR      5->180d    110053.13     -221374.90
4       GBP       1-0-7d     43017.94       43017.94
5       GBP      2-7-30d    186252.81      229270.75
6       GBP     3-30-90d   -104528.43      124742.32
7       GBP    4-90-180d   -284002.57     -159260.25
8       GBP      5->180d    123358.94      -35901.31
9       JPY       1-0-7d     36965.83       36965.83
10      JPY      2-7-30d    -22751.52       14214.31
11      JPY     3-30-90d    351562.01      365776.32
12      JPY    4-90-180d    -17767.18      348009.14
13      JPY      5->180d      2658.12      350667.26
14      USD     3-30-90d   -290661.22     -290661.22
15      USD    4-90-180d    168749.69     -121911.53
16      eur     3-30-90d    196503.91      196503.91
17      eur    4-90-180d    -65613.30      130

## Ahora resuelve en Pandas

In [105]:
# TODO: tu turno -- resuélvelo en pandas, sin mirar la solución

# Import data
df_transactions = pd.read_sql("SELECT * FROM transactions", conn)
df_transactions["MaturityDate"]=pd.to_datetime(df_transactions["MaturityDate"])

# Paso 1: I create a bucket list
bins=[-np.inf,7,30,90,180,np.inf] # Always len(bins)=len(labels)+1
labels=['1: 0-7d','2: 7-30d','3:30-90d','4:90-180d','5:>180d']
df_transactions["days_maturity"]=(df_transactions["MaturityDate"]-pd.Timestamp('2026-08-26')).dt.days
df_transactions["tenor_bucket"]=pd.cut(df_transactions["days_maturity"], bins=bins, labels=labels, ordered=True)

# Paso 2: Convert lower letters into capital letters
df_transactions["Currency"]=df_transactions["Currency"].str.upper()
#(df_transactions.head())


# df_transactions_new=df_transactions.dropna(subset=["Amount"]).groupby(["Currency","tenor_bucket"])["Amount"].sum().reset_index(name="SumByBucket")
# print(df_transactions_new)
# df_transactions_new=df_transactions_new.groupby(["Currency"])["SumByBucket"].cumsum().reset_index(name="AccumulatedByBucket")
# print(df_transactions_new)
print("----df_transactions----")
print(df_transactions.head())
# Paso 3: Now we group by tenor bucket and by currency
df_transactions_new=df_transactions.dropna(subset=["Amount"]).groupby(["Currency","tenor_bucket"],as_index=False)["Amount"].sum()
print("----df_transactions_new----")
print(df_transactions_new.head())
df_transactions_new=df_transactions_new.rename(columns={"Amount":"neto_bucket"})
df_transactions_new["Accummulated"]=df_transactions_new.groupby(["Currency"])["neto_bucket"].cumsum()
print("----df_transactions_new end----")
print(df_transactions_new.head())





----df_transactions----
   TransID Currency     Amount  CounterpartyID MaturityDate Notes  days_maturity tenor_bucket
0     1001      JPY   36965.83               1   2026-09-02  None              7      1: 0-7d
1     1002      USD -181419.83               1   2026-11-29  None             95    4:90-180d
2     1003      GBP   43017.94               1   2026-06-12  None            -75      1: 0-7d
3     1004      USD -131790.35               1   2026-09-23  None             28     2: 7-30d
4     1005      USD -173979.36               4   2026-11-16  None             82     3:30-90d
----df_transactions_new----
  Currency tenor_bucket     Amount
0      EUR     2: 7-30d  -69867.87
1      EUR     3:30-90d  -65056.25
2      EUR    4:90-180d  -65613.30
3      EUR      5:>180d  110053.13
4      GBP      1: 0-7d   43017.94
----df_transactions_new end----
  Currency tenor_bucket  neto_bucket  Accummulated
0      EUR     2: 7-30d    -69867.87     -69867.87
1      EUR     3:30-90d    -65056.25    

**Solución 3.2**


In [72]:
df_transactions = pd.read_sql("SELECT * FROM transactions", conn, parse_dates=["MaturityDate"])

hoy = pd.Timestamp("2026-08-26")
df_transactions["dias_hasta_vencimiento"] = (df_transactions["MaturityDate"] - hoy).dt.days

bins = [-np.inf, 7, 30, 90, 180, np.inf]  # -inf para capturar también vencimientos ya pasados
labels = ["0-7d", "7-30d", "30-90d", "90-180d", ">180d"]
df_transactions["tenor_bucket"] = pd.cut(
    df_transactions["dias_hasta_vencimiento"], bins=bins, labels=labels, ordered=True
)

# Paso 1: neto por divisa y bucket
neto = (
    df_transactions.dropna(subset=["Amount"])
    .groupby(["Currency", "tenor_bucket"], observed=False)["Amount"]
    .sum()
)

# Paso 2: acumulado dentro de cada divisa, respetando el orden cronológico de los buckets
gap_acumulado = neto.groupby(level="Currency").cumsum()

print(gap_acumulado)
print(neto)


Currency  tenor_bucket
EUR       0-7d                 0.00
          7-30d           -69867.87
          30-90d         -331428.03
          90-180d        -331428.03
          >180d          -221374.90
GBP       0-7d             43017.94
          7-30d           229270.75
          30-90d          124742.32
          90-180d        -159260.25
          >180d           -35901.31
JPY       0-7d             36965.83
          7-30d            14214.31
          30-90d          365776.32
          90-180d         348009.14
          >180d           350667.26
USD       0-7d                 0.00
          7-30d                0.00
          30-90d         -290661.22
          90-180d        -121911.53
          >180d          -121911.53
eur       0-7d                 0.00
          7-30d                0.00
          30-90d          196503.91
          90-180d         130890.61
          >180d           130890.61
usd       0-7d                 0.00
          7-30d          -263580.70
     

> Fíjate en una diferencia con la versión SQL: aquí usamos `observed=False` en el `groupby`, así que **todas** las combinaciones divisa/bucket aparecen, incluso las que no tienen ninguna transacción (el neto de ese bucket es 0, y el acumulado simplemente repite el valor del bucket anterior). En la Solución 3.1, esas combinaciones directamente no salían en el resultado, porque un `GROUP BY` en SQL nunca genera filas para grupos vacíos. Ambas cosas son correctas — solo cambia si prefieres ver la "foto completa" (pandas) o solo lo que realmente hay (SQL).

---
## Parte 4 — Lo mismo, pero con DuckDB sobre un DataFrame

Ahora practica el escenario alternativo: los datos ya están en un DataFrame de pandas (no en una base de datos), y quieres el mismo resultado usando SQL directamente sobre él.

In [ ]:
df = pd.read_sql("SELECT * FROM transactions", conn, parse_dates=["MaturityDate"])
df.head()


### Ejercicio 4.1
Repite el gap ladder completo del Ejercicio 3.1, pero con `duckdb.sql(...)` sobre el DataFrame `df` en vez de `pd.read_sql` sobre la conexión SQLite. La lógica SQL es prácticamente idéntica -- el cambio real está en qué motor ejecuta la consulta.

In [ ]:
# TODO
query_duckdb = """
-- tu consulta aquí (puedes reusar casi la misma lógica del Ejercicio 3.1)
"""
# resultado_duckdb = duckdb.sql(query_duckdb).df()
# print(resultado_duckdb)


**Solución 4.1**

In [ ]:
query_duckdb = """
WITH clasificado AS (
    SELECT
        Currency,
        Amount,
        CASE
            WHEN date_diff('day', DATE '2026-08-26', CAST(MaturityDate AS DATE)) <= 7 THEN '1-0-7d'
            WHEN date_diff('day', DATE '2026-08-26', CAST(MaturityDate AS DATE)) <= 30 THEN '2-7-30d'
            WHEN date_diff('day', DATE '2026-08-26', CAST(MaturityDate AS DATE)) <= 90 THEN '3-30-90d'
            WHEN date_diff('day', DATE '2026-08-26', CAST(MaturityDate AS DATE)) <= 180 THEN '4-90-180d'
            ELSE '5->180d'
        END as tenor_bucket
    FROM df
    WHERE Amount IS NOT NULL
),
agregado AS (
    SELECT Currency, tenor_bucket, SUM(Amount) as neto_bucket
    FROM clasificado
    GROUP BY Currency, tenor_bucket
)
SELECT
    Currency,
    tenor_bucket,
    neto_bucket,
    SUM(neto_bucket) OVER (PARTITION BY Currency ORDER BY tenor_bucket) as gap_acumulado
FROM agregado
ORDER BY Currency, tenor_bucket
"""
resultado_duckdb = duckdb.sql(query_duckdb).df()
print(resultado_duckdb)

conn.close()


> Único cambio real: `julianday()` (SQLite) se convirtió en `date_diff('day', ...)` (sintaxis de DuckDB) -- de nuevo, una diferencia de "dialecto" de fechas entre motores, no de lógica. El resto de la consulta (CTEs, CASE WHEN, PARTITION BY) es idéntico.

---
## Resumen del Día 4 — lo que deberías dominar ahora
- Orden real de ejecución de una query SQL en tu cabeza: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY.
- `HAVING` para filtrar después de agregar (a diferencia de `WHERE`, que filtra antes).
- `CASE WHEN` para clasificar filas en categorías -- el equivalente SQL de `pd.cut()` y `np.where()`.
- Funciones de ventana (`SUM() OVER (PARTITION BY ... ORDER BY ...)`) -- el equivalente exacto de `cumsum()`, y por qué necesitas prefijar los buckets con números si quieres que se ordenen cronológicamente.
- CTEs (`WITH ... AS`) para estructurar una consulta en pasos legibles, en vez de anidar subconsultas.
- El mismo gap ladder, construido dos veces -- con `sqlite3`/`pd.read_sql` y con DuckDB -- para que no dependas de adivinar qué te van a dar el lunes.

Con esto tienes cubierto de principio a fin, en pandas Y en SQL, el tipo de tarea que describía la persona de Ebury: construir datasets de liquidez automatizados, con lógica reproducible. Si te queda tiempo antes del lunes, el mejor uso que le puedes dar es repetir estos ejercicios sin mirar las soluciones, cronometrándote.